[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/29-capstone.ipynb)

# Module 29: Capstone: Build, Upgrade, Measure, Ship a Tiny GPT

Companion notebook to [Module 29](https://transformers-and-math.drmhse.com/modules/29-capstone) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

This notebook is a starting point for the capstone, not a finished answer. It contains a configurable character-level GPT, a training loop that records everything the report needs, and helpers for each part. The tasks themselves are in the module; the cells marked **TODO** are yours.

Keep `STEPS` small while you build, then raise it for the runs you report.

In [ ]:
import math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

STEPS = 300            # raise to 2,000+ for the runs you report
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
try:
    text = urllib.request.urlopen(URL, timeout=10).read().decode()
except Exception:
    text = ('First Citizen:\nBefore we proceed any further, hear me speak.\n\n' * 2000)
chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print(f'{len(text):,} characters, vocabulary {len(chars)}, device {DEVICE}')

## A configurable model
Every switch of section 15.6 is a flag, so each change in Part 29.2 is a one-line ablation.

In [ ]:
class Cfg:
    def __init__(self, **kw):
        self.d, self.h, self.L, self.T = 64, 4, 2, 64
        self.pre_ln = False; self.rope = False; self.rmsnorm = False; self.swiglu = False; self.kv_heads = None
        self.__dict__.update(kw)

class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__(); self.g = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * self.g

def rope(x):                                   # x: (B, H, T, dh)
    T, dh = x.shape[-2:]
    theta = 10000 ** (-torch.arange(0, dh, 2, device=x.device) / dh)
    ang = torch.arange(T, device=x.device)[:, None] * theta
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * ang.cos() - x2 * ang.sin(), x1 * ang.sin() + x2 * ang.cos()], -1).flatten(-2)

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c, self.hkv, self.dh = c, c.kv_heads or c.h, c.d // c.h
        self.q = nn.Linear(c.d, c.d); self.kv = nn.Linear(c.d, 2 * self.hkv * self.dh); self.o = nn.Linear(c.d, c.d)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.c.h, self.dh).transpose(1, 2)
        k, v = self.kv(x).view(B, T, 2, self.hkv, self.dh).permute(2, 0, 3, 1, 4)
        if self.c.rope:
            q, k = rope(q), rope(k)
        rep = self.c.h // self.hkv
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        return self.o(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, -1))

class FFN(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.swiglu = c.swiglu
        hidden = int(8 * c.d / 3) if c.swiglu else 4 * c.d
        self.up = nn.Linear(c.d, hidden); self.down = nn.Linear(hidden, c.d)
        self.gate = nn.Linear(c.d, hidden) if c.swiglu else None
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x) if self.swiglu else F.gelu(self.up(x)))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        Norm = RMSNorm if c.rmsnorm else nn.LayerNorm
        self.pre = c.pre_ln; self.n1, self.n2 = Norm(c.d), Norm(c.d)
        self.attn, self.ffn = Attention(c), FFN(c)
    def forward(self, x):
        if self.pre:
            x = x + self.attn(self.n1(x)); return x + self.ffn(self.n2(x))
        x = self.n1(x + self.attn(x)); return self.n2(x + self.ffn(x))

class GPT(nn.Module):
    def __init__(self, c, V):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(V, c.d)
        self.pos = None if c.rope else nn.Embedding(c.T, c.d)
        self.blocks = nn.Sequential(*[Block(c) for _ in range(c.L)])
        self.norm = (RMSNorm(c.d) if c.rmsnorm else nn.LayerNorm(c.d)) if c.pre_ln else nn.Identity()
        self.head = nn.Linear(c.d, V)
    def forward(self, ids):
        x = self.tok(ids)
        if self.pos is not None:
            x = x + self.pos(torch.arange(ids.shape[1], device=ids.device))
        return self.head(self.norm(self.blocks(x)))

## Training and evaluation helpers

In [ ]:
def batch(src, B=32, T=64):
    ix = torch.randint(len(src) - T - 1, (B,))
    x = torch.stack([src[i:i + T] for i in ix]); y = torch.stack([src[i + 1:i + T + 1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

@torch.no_grad()
def evaluate(model, src=None, batches=20):
    model.eval(); torch.manual_seed(1234)
    losses = [F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item() for x, y in (batch(src if src is not None else val_data) for _ in range(batches))]
    model.train()
    return sum(losses) / len(losses)

def run(cfg, seed, steps=STEPS, lr=3e-3):
    torch.manual_seed(seed)
    model = GPT(cfg, len(chars)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    curve = []
    for step in range(steps):
        x, y = batch(train_data)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if step % 100 == 0 or step == steps - 1:
            curve.append((step, loss.item()))
    val = evaluate(model)
    return dict(model=model, val=val, bits_per_char=val / math.log(2), params=sum(p.numel() for p in model.parameters()), curve=curve)

## 29.1 Build: the baseline with an error bar

In [ ]:
results = [run(Cfg(), seed) for seed in range(3)]
vals = torch.tensor([r['val'] for r in results])
print(f'baseline validation loss over 3 seeds: {[round(v, 4) for v in vals.tolist()]}  mean {vals.mean():.4f}  spread {vals.max() - vals.min():.4f}')
print(f'parameters: {results[0]["params"]:,}   formula 12·L·d² + |V|·d = {12 * 2 * 64 ** 2 + len(chars) * 64:,}')

## 29.2 Upgrade: one change at a time
**TODO.** Run each configuration over three seeds, keep everything else fixed, and compare with the spread above.

In [ ]:
LADDER = [
    ('2017 baseline', dict()),
    ('+ Pre-LN', dict(pre_ln=True)),
    ('+ RoPE', dict(pre_ln=True, rope=True)),
    ('+ RMSNorm', dict(pre_ln=True, rope=True, rmsnorm=True)),
    ('+ SwiGLU', dict(pre_ln=True, rope=True, rmsnorm=True, swiglu=True)),
    ('+ GQA (2 KV heads)', dict(pre_ln=True, rope=True, rmsnorm=True, swiglu=True, kv_heads=2)),
]
for name, kw in LADDER[:2]:                    # TODO: run the whole ladder, with 3 seeds each
    r = run(Cfg(**kw), seed=0)
    print(f'{name:22} val {r["val"]:.4f}   params {r["params"]:,}')

## 29.3 Evaluate, 29.4 Adapt, 29.5 Ship, 29.6 Explain
**TODO.** Each needs a few cells. Starting points:

- **Evaluate:** `evaluate(model, src=other_text_tensor)` gives the loss on any text; divide by `math.log(2)` for bits per character. Build your 50-prompt task suite as a list of `(prompt, expected)` pairs.
- **Adapt:** reuse `LoRALinear` from the Module 20 notebook, wrapping `block.attn.q` and `block.attn.o`.
- **Ship:** reuse `quantize` from the Module 24 notebook on every `nn.Linear` weight; reuse the cached decoder of the Module 12 notebook to test a KV cache with teacher forcing.
- **Explain:** read attention weights by computing `softmax(q kᵀ / √d_h)` from a block's `attn.q` and `attn.kv` outputs, and test a head by zeroing its slice of the attention output, as in the Module 14 notebook.